In [1]:
from typing import TypedDict,List,Dict

In [2]:
!pip install langchain-ollama

In [3]:
from langchain_ollama.llms import OllamaLLM

In [4]:
class State(TypedDict):
    messages: List[Dict[str,str]]

In [5]:
from langgraph.graph import StateGraph,START,END
graph_builder = StateGraph(State)

In [6]:
llm = OllamaLLM(model="llama3.1")

In [7]:
def chatbot(state: State):
    response = llm.invoke(state["messages"])
    state["messages"].append({"role": "assistant", "content":response})
    return {"messages":state["messages"]}
    

In [8]:
graph_builder.add_node("chatbott", chatbot)
graph_builder.add_edge(START,"chatbott")
graph_builder.add_edge("chatbott", END)

In [9]:
graph= graph_builder.compile()

In [10]:
def stream_graph_updates(user_input:str):
    state = {"messages": [{"role": "user", "content":user_input}]}
    for event in graph.stream(state):
       for value in event.values():
           print("Assitant", value["messages"][-1]["content"])    

In [ ]:
# Run chat bot in a loop
if __name__ == "__main__":
    while True:
        try:
            user_input = input("user: ")
            if user_input.lower() in ["quit", "exit", "q"]:
                print("Goodbye!")
                break
            
            stream_graph_updates(user_input)
        except Exception as e:
            print("An error occurred:", e)
            break